# Elastic Net Regularization

Elastic Net blends **L1 (Lasso)** and **L2 (Ridge)** penalties:

$$J(\beta) = \|\mathbf{X}\beta - y\|^{2} + \alpha\left(\rho\,\|\beta\|_{1} + (1-\rho)\,\|\beta\|_{2}^{2}\right)$$

- `alpha` (a) controls overall strength of regularization.
- `l1_ratio` (rho): `1` = pure Lasso, `0` = pure Ridge, in-between = mix.

Elastic Net is preferred when features are **correlated** (Lasso tends to pick one at random) and when you still want some feature selection.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.preprocessing import StandardScaler

# Same synthetic dataset as the Ridge / Lasso examples
np.random.seed(42)
n = 30
size = np.random.uniform(500, 3000, n)
rooms = size / 400 + np.random.randn(n) * 1.5
noise_feat = np.random.randn(n)
X = np.column_stack([size, rooms, noise_feat])
y = 100 * size + 50 * rooms + np.random.randn(n) * 2000
feature_names = ["size", "rooms", "noise_feat"]


## Fit OLS vs. Elastic Net (50/50 mix)


In [ ]:
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

ols = LinearRegression().fit(Xs, y)
enet = ElasticNet(alpha=300.0, l1_ratio=0.5, max_iter=10000).fit(Xs, y)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "OLS": np.round(ols.coef_, 2),
    "ElasticNet (a=300, r=0.5)": np.round(enet.coef_, 2),
})
print(coef_df.to_string(index=False))
print(f"\nIntercept  OLS : {ols.intercept_:.2f}")
print(f"Intercept EN  : {enet.intercept_:.2f}")


## Effect of l1_ratio (0 = Ridge, 1 = Lasso)


In [ ]:
ratios = [0.0, 0.5, 1.0]
labels = ["Ridge (r=0)", "Mix (r=0.5)", "Lasso (r=1)"]
colors = ["teal", "purple", "crimson"]

plt.figure(figsize=(8, 4.5))
for r, lab, c in zip(ratios, labels, colors):
    m = ElasticNet(alpha=300.0, l1_ratio=r, max_iter=10000).fit(Xs, y)
    plt.plot(feature_names, m.coef_, marker="o", label=lab, color=c)
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Coefficient (standardized)")
plt.title("Elastic Net: sweep of l1_ratio")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## Takeaways

- `l1_ratio = 0` -> identical to **Ridge**; `l1_ratio = 1` -> identical to **Lasso**.
- The mixed case keeps useful correlated features (both `size` and `rooms`) while still shrinking the noise feature.
- Use `ElasticNetCV` to tune both `alpha` and `l1_ratio` together.
